# Elasticity regressions

In [ ]:
import linearmodels as lm
import numpy as np
import pandas as pd
import statsmodels.api as sm

from Smn.config import PROCESSED_DATA_DIR
from Smn.utils import create_lags, plot_nformat, resample_nlog

clean_df = pd.read_csv(
    PROCESSED_DATA_DIR / "clean_df.csv", parse_dates=["date", "created_date"]
)

### Pooled regressions

In [ ]:
daily_df = resample_nlog(clean_df)
plot_nformat(daily_df)

In [ ]:
weekly_df = resample_nlog(clean_df, "W")
plot_nformat(weekly_df)

### Lagged OLS — do past discounts predict today's sales?

In [ ]:
daily_lags = create_lags(daily_df, 5, "log_discount")

lag_cols = ["log_discount"] + [f"log_discount_lag{i}" for i in range(1, 6)]
X = sm.add_constant(daily_lags[lag_cols])
y = daily_lags["log_sales"]

model = sm.OLS(y, X).fit()
model.summary()

### Fixed effects regressions

Panel OLS with entity effects to control for unobserved heterogeneity across groups.

In [ ]:
def panel_data(df, group_col, freq="d"):
    """Resample within groups, returning a MultiIndex for PanelOLS."""
    grouped = (
        df.groupby(group_col)
        .resample(freq, on="date")
        .agg(
            sales=("product_quantity", "sum"),
            price=("unit_price", "mean"),
            discount=("discount", "mean"),
        )
    )
    grouped["log_sales"] = np.log(grouped["sales"] + 1)
    grouped["log_discount"] = np.log(grouped["discount"] + 1)
    return grouped

In [ ]:
# by price category
grouped_price = panel_data(clean_df, "price_category")

model_price = lm.PanelOLS.from_formula(
    "log_sales ~ log_discount + EntityEffects", data=grouped_price
)
model_price.fit()

In [ ]:
# by brand
grouped_brand = panel_data(clean_df, "long")

model_brand = lm.PanelOLS.from_formula(
    "log_sales ~ log_discount + EntityEffects", data=grouped_brand
)
model_brand.fit()